# RAG demo — search, evidence, and the agent

This notebook shows the **citation layer** on top of hybrid retrieval:

1. `search_docs()` wraps `HybridRetriever` as `RagEvidence` (chunk id, title, page, excerpt).
2. `run_agent()` routes a question, retrieves evidence, and returns structured claims that cite `E1`, `E2`, …

Low-level BM25 vs dense vs RRF comparison lives in `query.ipynb`.

**Prerequisites:** run `scripts/build_index.py`. Agent cells need `FIREWORKS_API_KEY` in `.env`.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

from IPython.display import Markdown, display

# Settings and indexes are resolved from the repo root.
PROJECT_ROOT = Path("..").resolve()
os.chdir(PROJECT_ROOT)

from wellground.config import get_settings
from wellground.retrieval import HybridRetriever
from wellground.tools.rag import search_docs

settings = get_settings()
retriever = HybridRetriever.load(
    bm25_dir=settings.wellground_bm25_path,
    chroma_dir=settings.wellground_chroma_path,
)
print(f"BM25 dir: {settings.wellground_bm25_path}")
print(f"Chunks: {len(retriever.bm25.chunks)}")

/Users/mehuljain/Documents/Projects/well/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6897.79it/s]


BM25 dir: data/processed/bm25
Chunks: 249


## Search reports

FORGE questions that should hit daily reports / inj-prod PDFs.

In [2]:
QUESTIONS = [
    "Summarize how 16A was stimulated",
    "What rig activity happened on Aug 12?",
    "Where does injection enter the 16A wellbore?",
]


def show_evidence(title: str, items) -> None:
    display(Markdown(f"### {title}"))
    if not items:
        display(Markdown("_No hits_"))
        return
    lines = []
    for item in items:
        wells = ", ".join(item.well_ids) or "—"
        lines.append(
            f"- **{item.chunk_id}** · {item.title} p.{item.page} · "
            f"wells={wells} · score={item.score:.4f}\n"
            f"  {item.excerpt[:240]}"
        )
    display(Markdown("\n".join(lines)))


for question in QUESTIONS:
    show_evidence(question, search_docs(question, retriever=retriever))

### Summarize how 16A was stimulated

- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.4, 8-1-2024#p001#c01** · FORGE 16B(78)-32 Clean Out RPT No.4, 8-1-2024 p.1 · wells=16B, 16A · score=0.0164
  0 EVAL Move over to 16A well and rig up and run pressure transducer and fill with water from minnion tank through flow cross. 10:00 14:00 4.00 10,947.0 WOE Work with Clay Jones on fittings for mini separator and sample ports on 16B flow lin
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024#p001#c01** · FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024 p.1 · wells=16A, 16B · score=0.0164
  Stop pumping on 16A(78)-32 and shut well in. Monitor pressures on 16B(78)-32. 10:00 - 307 psi 12:00 - 280 psi 14:00 - 258 psi 16:00 - 242 psi 17:00 - 236 psi 17:00 1:00 8.00 10,947.0 EVAL Open well to separator. Pumping 2.5 bpm down 16A, St
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024#p001#c00** · FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024 p.1 · wells=16B · score=0.0161
  22Report No: Report For 06:00 AM 28-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: 30 Day Flow Test 20 / 160Total:4 / 24Other:14 
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01** · FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024 p.1 · wells=16A, 16B · score=0.0161
  80 psi, T 278°F, F 121 gpm 13:00 - P 80 psi, T 282°F, F 122 gpm 13:00 6:00 17.00 10,947.0 EVAL Increase pump rate on 16A(78)-32 to 7.5 bpm, Step 6 of pump schedule. 13:30 - P 82 psi, T 282°F, F 125 gpm 15:30 - P 92 psi, T 290°F, F 86 gpm 17
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024[1]#p001#c00** · FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024[1] p.1 · wells=16B · score=0.0159
  22Report No: Report For 06:00 AM 28-Aug-24 Daily Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: 30 Day Flow Test 20 / 160Total:4 / 24Other:14 
- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT5, 8-10-2024#p001#c01** · FORGE 16A(78)-32 Circulation Test RPT5, 8-10-2024 p.1 · wells=16A, 16B · score=0.0159
  pressures for 8 hours. 10:00 - 209 psi 12:00 - 186 psi 14:00 - 176 psi 16:00 - 168 psi 17:00 - 168 psi 17:00 1:00 8.00 10,987.0 PUMP Open 16A well, check all lines, open 16B(78)-32 well to separator and begin pumping Step 4 of pump schedule

### What rig activity happened on Aug 12?

- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT No.1, 8-6-2024#p001#c00** · FORGE 16A(78)-32 Circulation Test RPT No.1, 8-6-2024 p.1 · wells=16A, 16B · score=0.0164
  1Report No: Report For 06:00 AM 06-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United States Job ID: Circulation Test 1 / 24Total:0 / 0Other:0 / 0
- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.9, 8-6-2024#p001#c01** · FORGE 16B(78)-32 Clean Out RPT No.9, 8-6-2024 p.1 · wells=16A, 16B · score=0.0164
  00 1.00 10,947.0 RIGD Rig down Weatherford power tongs and set off floor. Release Weatherford hand. Release UDES pump truck. 11:00 14:00 3.00 10,947.0 RIGD Begin rigging down UDES WOR 117. Nipple down Washington head and remove from stack. 
- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT2, 8-7-2024#p001#c00** · FORGE 16A(78)-32 Circulation Test RPT2, 8-7-2024 p.1 · wells=16A, 16B · score=0.0161
  2Report No: Report For 06:00 AM 07-Aug-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United States Job ID: Circulation Test 5 / 72Total:0 / 0Other:4 / 4
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT16, 8-22-2024#p001#c02** · FORGE 16B(78)-32 Circulation Test RPT16, 8-22-2024 p.1 · wells=16B · score=0.0161
  500 38 OTHER Bulk InventoryWater / Fluids InventoryWater / Fluids HauledSafety Information First Aid Treatments: 0 0 0Medical Treatments: Lost Time Incidents: Days Since LTI: BOP Test Crownamatic Check Weather Information Sky Condition: Air
- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.5, 8-2-2024#p001#c00** · FORGE 16B(78)-32 Clean Out RPT No.5, 8-2-2024 p.1 · wells=16B · score=0.0159
  5Report No: Report For 06:00 AM 02-Aug-24 Daily Completion Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: Clean Out Original WellboreWellbore:
- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT10, 8-15-2024#p001#c01** · FORGE 16A(78)-32 Circulation Test RPT10, 8-15-2024 p.1 · wells=16A · score=0.0159
  11:30 - 2,875 psi 13:30 - 2,866 psi 15:30 - 2,925 psi 17:30 - 2,929 psi 19:30 - 2,944 psi 21:30 - 2,931 psi 23:30 - 2,936 psi 01:30 - 2,935 psi 03:30 - lost communication 06:00 - 2,933 psi Management Summary Continue pumping Step 7 on pump 

### Where does injection enter the 16A wellbore?

- **inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p003#c00** · University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report p.3 · wells=16A · score=0.0305
  University of Utah Forge 16A(78)-32 Objectives: Quantitative downhole injection profile. Well / Job Information: Casing: 7”, 38#, T-95 @ 10787’ MD Open Hole: 8-3/4” OH @ 10987’ MD Tubing: None KOP @ 5957’ MD / 5955.66’ TVD / 5.67 degrees de
- **inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report#p002#c00** · University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Final Report p.2 · wells=16A · score=0.0164
  University of Utah Forge 16A(78)-32 LEGAL DISCLAIMER SAVE FOR THAT EXPRESSLY STATED IN THE AGREEMENT, SLB MAKES NO WARRANTIES, EITHER EXPRESS OR IMPLIED, STATUTORY OR OTHERWISE IN CONNECTION WITH ITS PERFORMANCE OF THE SERVICES OR ANY DELIV
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024#p001#c01** · FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024 p.1 · wells=16A, 16B, 58-32 · score=0.0164
  of pump schedule. Vary rates as necessary while running the EV camera in 16A well. Return to 10 bpm when job complete. Monitor pressure, temperature and flow on 16B. Separator rocking back and forth. Open bypass line from well to sump to re
- **inj_prod/16A/University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Report Prelim#p002#c00** · University of Utah_Forge16A(78)-32_SLB_17Aug2024_Injection Profile Report Prelim p.2 · wells=16A · score=0.0161
  University of Utah Forge 16A(78)-32 LEGAL DISCLAIMER SAVE FOR THAT EXPRESSLY STATED IN THE AGREEMENT, SLB MAKES NO WARRANTIES, EITHER EXPRESS OR IMPLIED, STATUTORY OR OTHERWISE IN CONNECTION WITH ITS PERFORMANCE OF THE SERVICES OR ANY DELIV
- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024#p001#c01** · FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024 p.1 · wells=16A, 16B · score=0.0161
  Stop pumping on 16A(78)-32 and shut well in. Monitor pressures on 16B(78)-32. 10:00 - 307 psi 12:00 - 280 psi 14:00 - 258 psi 16:00 - 242 psi 17:00 - 236 psi 17:00 1:00 8.00 10,947.0 EVAL Open well to separator. Pumping 2.5 bpm down 16A, St
- **inj_prod/16B/University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report#p002#c00** · University of Utah_Forge16B(78)-32_SLB_28Aug2024_DiDrill_Production Log Final Report p.2 · wells=16B · score=0.0159
  University of Utah Forge 16B(78)-32 LEGAL DISCLAIMER SAVE FOR THAT EXPRESSLY STATED IN THE AGREEMENT, SLB MAKES NO WARRANTIES, EITHER EXPRESS OR IMPLIED, STATUTORY OR OTHERWISE IN CONNECTION WITH ITS PERFORMANCE OF THE SERVICES OR ANY DELIV

## Raw hybrid hits vs `RagEvidence`

Same ranking, extra citation fields (`chunk_id`, page, wells) ready for the synthesizer.

In [3]:
COMPARE_Q = QUESTIONS[0]
hits = retriever.search(COMPARE_Q, top_k=6)
evidence = search_docs(COMPARE_Q, retriever=retriever, top_k=6)

rows = ["| rank | hybrid chunk_id | evidence chunk_id | page | score |"]
rows.append("|---|---|---|---|---|")
for hit, item in zip(hits, evidence, strict=True):
    rows.append(
        f"| {hit.rank} | `{hit.chunk_id}` | `{item.chunk_id}` | {item.page} | {hit.score:.4f} |"
    )
display(Markdown(f"**Query:** {COMPARE_Q}\n\n" + "\n".join(rows)))

**Query:** Summarize how 16A was stimulated

| rank | hybrid chunk_id | evidence chunk_id | page | score |
|---|---|---|---|---|
| 1 | `daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.4, 8-1-2024#p001#c01` | `daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.4, 8-1-2024#p001#c01` | 1 | 0.0164 |
| 2 | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024#p001#c01` | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT4, 8-10-2024#p001#c01` | 1 | 0.0164 |
| 3 | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024#p001#c00` | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024#p001#c00` | 1 | 0.0161 |
| 4 | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT5, 8-11-2024#p001#c01` | 1 | 0.0161 |
| 5 | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024[1]#p001#c00` | `daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT22, 8-28-2024[1]#p001#c00` | 1 | 0.0159 |
| 6 | `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT5, 8-10-2024#p001#c01` | `daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT5, 8-10-2024#p001#c01` | 1 | 0.0159 |

## Agent RAG path

Router → RAG worker → synthesizer → verifier. Output is structured JSON (`claims` + `evidence`). Requires Fireworks.

In [10]:
from wellground.agent.graph import run_agent

if not settings.fireworks_api_key:
    print("Set FIREWORKS_API_KEY in .env to run the agent.")
else:
    response = run_agent("When was the wellbore cleared and rigged for testing??")
    display(Markdown(f"**status:** `{response.status}` · **route:** `{response.route}`"))
    if response.refusal_reason:
        display(Markdown(f"**refusal:** {response.refusal_reason}"))
    claim_lines = [
        f"- {claim.text}  \n  cites: `{', '.join(claim.source_ids)}`"
        for claim in response.claims
    ]
    display(Markdown("### Claims\n" + ("\n".join(claim_lines) or "_(none)_")))
    display(Markdown("### Evidence"))
    show_evidence("cited passages", [e for e in response.evidence if e.kind == "rag"])
    display(Markdown("### Raw JSON"))
    print(response.model_dump_json(indent=2))

{"run_id": "fc808c28-2cad-48b0-85e8-56857e428fbb", "route": "rag", "status": "refused", "evidence_count": 6, "latency_ms": 8503.5, "model": "accounts/fireworks/models/gpt-oss-120b", "question": "When was the wellbore cleared and rigged for testing??"}


**status:** `refused` · **route:** `rag`

**refusal:** claim[0] is not grounded in cited evidence; claim[1] is not grounded in cited evidence

### Claims
_(none)_

### Evidence

### cited passages

- **daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024#p001#c01** · FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024 p.1 · wells=16A, 16B, 58-32 · score=0.0164
  of pump schedule. Vary rates as necessary while running the EV camera in 16A well. Return to 10 bpm when job complete. Monitor pressure, temperature and flow on 16B. Separator rocking back and forth. Open bypass line from well to sump to re
- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.2, 7-30-2024#p001#c00** · FORGE 16B(78)-32 Clean Out RPT No.2, 7-30-2024 p.1 · wells=16B, 16A · score=0.0164
  2Report No: Report For 06:00 AM 30-Jul-24 Daily Completion Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: Clean Out Original WellboreWellbore:
- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT31, 9-5-2024#p001#c00** · FORGE 16A(78)-32 Circulation Test RPT31, 9-5-2024 p.1 · wells=16A · score=0.0161
  31Report No: Report For 06:00 AM 05-Sep-24 Daily Report Well ID: Utah FORGE 16A(78)-32-STIM1 Well Name: FORGE 16A(78)-32-STIM1 Field: FORGE County: Beaver State: UT Country: United States Job ID: Circulation Test 0 / 0Total:0 / 0Other:0 / 0
- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.9, 8-6-2024#p001#c00** · FORGE 16B(78)-32 Clean Out RPT No.9, 8-6-2024 p.1 · wells=16A, 16B · score=0.0161
  9Report No: Report For 06:00 AM 06-Aug-24 Daily Completion Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: Clean Out Original WellboreWellbore:
- **daily_reports/extracted/FORGE 16A(78)-32 Circulation Test RPT30, 9-4-2024#p001#c01** · FORGE 16A(78)-32 Circulation Test RPT30, 9-4-2024 p.1 · wells=16A, 16B · score=0.0159
  ,810 psi 15:30 - 2,801 psi 17:30 - 2,793 psi 19:30 - 2,792 psi 21:30 - 2,810 psi 23:30 - 2,812 psi 01:30 - 2,813 psi 03:30 - 2,802 psi 05:30 - 2,819 psi Pason hooking up 2" flow meter on discharge line to sump for CSULB testing. Checked out
- **daily_reports/extracted/FORGE 16B(78)-32 Clean Out RPT No.5, 8-2-2024#p001#c00** · FORGE 16B(78)-32 Clean Out RPT No.5, 8-2-2024 p.1 · wells=16B · score=0.0159
  5Report No: Report For 06:00 AM 02-Aug-24 Daily Completion Report Well ID: Utah FORGE FORGE 16B(78)-32 Well Name: FORGE 16B(78)-32 Field: FORGE Sect: 32 Town: 26S Rng: 9W County: BEAVER State: UT Job ID: Clean Out Original WellboreWellbore:

### Raw JSON

{
  "status": "refused",
  "route": "rag",
  "claims": [],
  "evidence": [
    {
      "kind": "rag",
      "evidence_id": "E1",
      "chunk_id": "daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024#p001#c01",
      "doc_id": "daily_reports/extracted/FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024",
      "title": "FORGE 16B(78)-32 Circulation Test RPT13, 8-19-2024",
      "page": 1,
      "excerpt": "of pump schedule. Vary rates as necessary while running the EV camera in 16A well. Return to 10 bpm when job complete. Monitor pressure, temperature and flow on 16B. Separator rocking back and forth. Open bypass line from well to sump to relieve pressure on separator. Rig up additional bypass line to sump. SLB spotted logging truck, moved lubricator and spotted crane. Began rigging up SLB tools. SureFire removed spool from top of wellhead and made up lubricator adaptor to top of crown valve. ...",
      "well_ids": [
        "16A",
        "16B",
        "58-32"